In [2]:
import pandas as pd
import torch
from sklearn.utils import shuffle
import pandas as pd

In [4]:
from os import chdir
from pathlib import Path

cwd = Path.cwd()
print(f"CWD: {cwd}")
if cwd.name == "Sota"or cwd.name == "ML100K":
    chdir("/Users/haowen/Documents/Decentral RS/fed-learning-main")
print(f"CWD: {Path.cwd()}")

CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main/code/Sota/ML100K
CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main


In [6]:
train = pd.read_csv("dataset/ml100k_train.csv")
test = pd.read_csv("dataset/ml100k_test.csv")
train.head()

,user_id,item_id,rating,rating.1
0,207,87,5,5
1,675,900,4,4
2,373,230,2,2
3,377,564,3,3
4,725,843,3,3


In [8]:
train.head()

,user_id,item_id,rating,rating.1
0,207,87,5,5
1,675,900,4,4
2,373,230,2,2
3,377,564,3,3
4,725,843,3,3


In [10]:
rating_matrix = train.pivot(index='user_id', columns='item_id', values='rating')
n_users, n_movies = rating_matrix.shape
# Scaling ratings to between 0 and 1, this helps our model by constraining predictions
min_rating, max_rating = train['rating'].min(), train['rating'].max()
rating_matrix = (rating_matrix - min_rating) / (max_rating - min_rating)
print(rating_matrix)
print(n_users*n_movies-rating_matrix.isnull().values.sum())

item_id  0     1     2     3     4     5     6     7     8     9     ...  \
user_id                                                              ...   
0        1.00   0.5  0.75  0.50   NaN   1.0   NaN   0.0   NaN  0.50  ...   
1         NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  0.25  ...   
2         NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
3         NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
4        0.75   0.5   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
...       ...   ...   ...   ...   ...   ...   ...   ...   ...   ...  ...   
938       NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   1.0   NaN  ...   
939       NaN   NaN   NaN  0.25   NaN   NaN  0.75   1.0   NaN   NaN  ...   
940       NaN   NaN   NaN   NaN   NaN   NaN  0.75   NaN   NaN   NaN  ...   
941       NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
942       NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   

item_id  16

In [12]:
n_users, n_movies

(943, 1628)

In [14]:
# Replacing missing ratings with -1 so we can filter them out later

rating_matrix[rating_matrix.isnull()] = -1
rating_matrix = torch.FloatTensor(rating_matrix.values)
print(rating_matrix)

tensor([[ 1.0000,  0.5000,  0.7500,  ..., -1.0000, -1.0000, -1.0000],
        [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
        [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
        ...,
        [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
        [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
        [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000]])


In [16]:
class Loss(torch.nn.Module):
    def __init__(self, lam_u=0.3, lam_v=0.3):
        super().__init__()
        self.lam_u = lam_u
        self.lam_v = lam_v
    
    def forward(self, matrix, u_features, v_features):
        non_zero_mask = (matrix != -1).type(torch.FloatTensor)
        predicted = torch.sigmoid(torch.mm(u_features, v_features.t()))
        
        diff = (matrix - predicted)**2
        prediction_error = torch.sum(diff*non_zero_mask)

        u_regularization = self.lam_u * torch.sum(u_features.norm(dim=1))
        v_regularization = self.lam_v * torch.sum(v_features.norm(dim=1))

        return prediction_error + u_regularization + v_regularization, prediction_error

In [18]:
# RFRecF: average movie_features 

alpha = 0.025             # learning rate 0.1, 0.01
threshold = 0.8         # prob of L2GD 0.8, 0.4
lam = 10
# lr = alpha/(1-threshold)  
# eta = min(1,alpha*lam/threshold)
lr = 0.1
eta = 1       
num_client = 200        
m = n_users//num_client
num_epoch = 200         # 100
latent_vectors = 20    
# user_features = torch.randn(n_users, latent_vectors, requires_grad=True)

user_features = []
movie_features = []
for i in range(num_client):
    user_features.append(torch.randn(m, latent_vectors, requires_grad=True))
    movie_features.append(torch.randn(n_movies, latent_vectors, requires_grad=True))
with torch.no_grad():
    for i in range(num_client):
        # print(user_features[i])
        user_features[i].data.mul_(0.01)        # 0.1
        movie_features[i].data.mul_(0.01)       # 0.1
        # print(user_features[i])
    # print(movie_features[i])
RFRecF_error = Loss(lam_u=0.1, lam_v=0)       # lam_u=0.1, lam_v=0

optimizer_client_set = []
optimizer_user_set = []
optimizer_movie_set = []

for i in range(num_client):
    optimizer_client = torch.optim.Adam([user_features[i], movie_features[i]], lr=lr) 
    optimizer_client_set.append(optimizer_client)

aver_movie_features = torch.randn(n_movies, latent_vectors).data.mul_(0.01)
# aver_movie_features = torch.zeros(n_movies, latent_vectors).data.mul_(0.01)
error_list = []
previous = torch.zeros(n_movies, latent_vectors)
for step, epoch in enumerate(range(num_epoch)):
    aver_loss = 0
    aver_prediction_error = 0
    
    # tmp = torch.zeros(n_movies, latent_vectors)
    p = torch.rand(1)
    # print(p)
    if p>threshold:
        # local update
        for i in range(num_client):
            optimizer_client_set[i].zero_grad()
  
            loss, prediction_error = RFRecF_error(rating_matrix[i*m:(i+1)*m], user_features[i], movie_features[i])
            aver_loss += loss/num_client
            aver_prediction_error += prediction_error/num_client

            loss.backward()
            # for params in optimizer_client_set[i].param_groups:                      
            #     params['lr'] = 0.1 
            optimizer_client_set[i].step()
    else:
        # server update
        tmp = torch.zeros(n_movies, latent_vectors)
        with torch.no_grad():
            for i in range(num_client):
   
                # # laplace mechanism
                # s = 0.08
                # delta = 0.2
                # noise = torch.distributions.laplace.Laplace(torch.tensor([0.0]), torch.tensor([s])).sample()
                # tmp+= torch.clip(movie_features[i], min=-delta, max=delta) + noise

                # without perturbation
                tmp+= movie_features[i]

            aver_movie_features = tmp/num_client
            for i in range(num_client):
                movie_features[i] += eta*(aver_movie_features - movie_features[i]) 

    if step % 1 == 0:
        # print(f"Step {step}, {aver_loss:.3f}")
        print(f"Step {step}, {aver_prediction_error:.3f}")

Step 0, 0.000
Step 1, 0.000
Step 2, 0.000
Step 3, 0.000
Step 4, 0.000
Step 5, 0.000
Step 6, 31.056
Step 7, 0.000
Step 8, 31.088
Step 9, 0.000
Step 10, 0.000
Step 11, 0.000
Step 12, 0.000
Step 13, 0.000
Step 14, 0.000
Step 15, 31.051
Step 16, 29.346
Step 17, 0.000
Step 18, 0.000
Step 19, 0.000
Step 20, 0.000
Step 21, 30.928
Step 22, 0.000
Step 23, 30.801
Step 24, 0.000
Step 25, 0.000
Step 26, 30.645
Step 27, 0.000
Step 28, 0.000
Step 29, 0.000
Step 30, 0.000
Step 31, 0.000
Step 32, 0.000
Step 33, 0.000
Step 34, 30.440
Step 35, 0.000
Step 36, 30.173
Step 37, 0.000
Step 38, 0.000
Step 39, 0.000
Step 40, 0.000
Step 41, 0.000
Step 42, 0.000
Step 43, 0.000
Step 44, 0.000
Step 45, 0.000
Step 46, 0.000
Step 47, 0.000
Step 48, 0.000
Step 49, 0.000
Step 50, 0.000
Step 51, 0.000
Step 52, 0.000
Step 53, 0.000
Step 54, 0.000
Step 55, 29.830
Step 56, 0.000
Step 57, 29.393
Step 58, 0.000
Step 59, 0.000
Step 60, 0.000
Step 61, 0.000
Step 62, 0.000
Step 63, 0.000
Step 64, 0.000
Step 65, 0.000
Step 66, 

In [19]:
test_rating_matrix = test.pivot(index='user_id', columns='item_id', values='rating')
n_users, n_movies = test_rating_matrix.shape
test_rating_matrix[test_rating_matrix.isnull()] = -1
test_rating_matrix = torch.FloatTensor(test_rating_matrix.values)
# test_rating_matrix

In [ ]:
train_items = train["item_id"].drop_duplicates().sort_values().reset_index(drop=True)
test_items = test["item_id"].drop_duplicates().sort_values().reset_index(drop=True)
# Create a mapping from item_id → index in train_items
item_to_index = pd.Series(train_items.index.values, index=train_items.values)

# Map test_items to train indices
test_indices_in_train = test_items.map(item_to_index)

print(test_indices_in_train)

In [22]:
n_users, n_movies

(943, 1408)

In [24]:
non_zero_mask = (test_rating_matrix != -1).type(torch.FloatTensor)
num = torch.sum(non_zero_mask)
def Error(matrix, u_features, v_features):
    predicted_ratings = torch.sigmoid(torch.mm(u_features,v_features.t()))
    predicted_ratings = predicted_ratings[:,test_indices_in_train]
    # predicted_ratings = non_zero_mask * predicted_ratings
    pred = (predicted_ratings*(max_rating - min_rating) + min_rating)*non_zero_mask[i*m:(i+1)*m]
    actual = matrix*non_zero_mask[i*m:(i+1)*m]
    AE_diff = torch.abs(pred - actual)
    SE_diff = (pred - actual)**2
    
    prediction_abs_error = torch.sum(AE_diff)
    prediction_squared_error = torch.sum(SE_diff)
    n_non_zero = torch.sum(non_zero_mask[i*m:(i+1)*m])
    return prediction_abs_error, prediction_squared_error, n_non_zero

AE_error = 0
SE_error = 0
num_non_zero = 0
movie_features = aver_movie_features

In [26]:
for i in range(num_client):
    AE_error += Error(test_rating_matrix[i*m:(i+1)*m], user_features[i], movie_features)[0]
    SE_error += Error(test_rating_matrix[i*m:(i+1)*m], user_features[i], movie_features)[1]
    num_non_zero += Error(test_rating_matrix[i*m:(i+1)*m], user_features[i], movie_features)[2]
test_MAE = AE_error/num_non_zero
test_RMSE = torch.sqrt(SE_error/num_non_zero)
print('test_MAE =', test_MAE.data.numpy())
print('test_RMSE =', test_RMSE.data.numpy())

NameError: name 'test_indices_in_train' is not defined

In [15]:
i =1
Error(test_rating_matrix[i*m:(i+1)*m], user_features[i], movie_features)

(tensor(186.6021, grad_fn=<SumBackward0>),
 tensor(240.8646, grad_fn=<SumBackward0>),
 tensor(219.))

In [16]:
movie_features

tensor([[ 2.3318e-01,  6.7642e-02,  5.8241e-02,  ..., -3.3935e-01,
          3.8956e-01, -1.4925e-01],
        [ 1.8840e-01,  5.5973e-03, -6.6424e-02,  ..., -1.3074e-01,
          9.1775e-06, -1.5961e-01],
        [ 4.0637e-02,  1.2695e-01, -2.6947e-02,  ..., -2.0338e-01,
          8.0980e-03, -7.6100e-02],
        ...,
        [-2.8324e-04,  1.2249e-04,  3.8444e-05,  ...,  9.4211e-04,
          1.5308e-04,  3.5079e-04],
        [ 1.4184e-03,  3.6595e-04,  6.0689e-04,  ..., -3.3908e-04,
         -6.1610e-04,  4.6917e-04],
        [ 1.1997e-03, -3.0686e-05,  4.5354e-04,  ...,  1.0253e-04,
         -1.6858e-03,  1.7758e-03]])